In [1]:
import torch
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import PCGP.gpytorch_tools as gt
from mpl_toolkits.mplot3d import Axes3D  # Needed for 3D plotting
from matplotlib import cm  # Colormaps
from matplotlib import gridspec
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import os
import pandas as pd
import seaborn as sns
plt.rcParams.update({'font.size': 22})
base_dir = os.getcwd()
print(base_dir)
save_dir = os.path.join(base_dir, "graphs")
output_data_ex2 = os.path.join(base_dir,  "Experiment2_Tripendulum", "output_data")
input_data_ex2 = os.path.join(base_dir,  "Experiment2_Tripendulum", "input_data")

/home/johanna/Documents/PhD/project/github/GhostTasking


# posteriors laplace

In [ ]:
mode = "GT"
run = 9
sigma_list = [0.0, 0.01, 0.1, 0.2]
npoints_list = [3, 5, 10, 25]
kernel_mode = "shared_kernel"
Al_mode = "withAl"
plt.rcParams.update({'font.size': 16})

fig, ax = plt.subplots(4,4, figsize = (12, 12))
N = 400
cmap = "jet"
for i in range(4):
    npoints = npoints_list[i]
    for j in range(4):
        sigma = sigma_list[j]
        data_ex2 = np.load(os.path.join(output_data_ex2, "shared_withAl", f"Ex2_{mode}_n{npoints}_sigma{sigma:.2f}_{run}.npz"))
        learned_parameters = data_ex2["learned_parameters"]
        true_parameters = data_ex2["true_parameters"]
        laplace_covariance = data_ex2["laplace_covariance"]
        r1 = max(0.5, 1.1*abs(learned_parameters[0]-true_parameters[0]))
        r2 = max(0.5, 1.1*abs(learned_parameters[1]-true_parameters[1]))
        xlim = (learned_parameters[0]-r1, learned_parameters[0]+r1)
        ylim = (learned_parameters[1]-r2, learned_parameters[1]+r2)
        
        mesh_1 = torch.linspace(xlim[0], xlim[1], N)
        mesh_2 = torch.linspace(ylim[0], ylim[1], N)
        param1, param2 = torch.meshgrid(mesh_1, mesh_2, indexing="xy")
        
        mu = learned_parameters
        inv_cov = torch.inverse(torch.tensor(laplace_covariance[:2,:2]))
        param_vec = torch.stack([param1.flatten(), param2.flatten()], dim=-1)
        exponent_raw = torch.einsum('bi,ij,bj->b', param_vec - mu, inv_cov, param_vec - mu)
        prob_density = torch.exp(-0.5*(exponent_raw - exponent_raw.min()))
        prob_density /= prob_density.sum()*(mesh_1[1]-mesh_1[0])*(mesh_2[1]-mesh_2[0])
        prob_density = prob_density.reshape(N,N)
        
        surf = ax[i,j].pcolormesh(
                param1, param2, prob_density,
                cmap=cmap,
                shading="auto"
            )
        
        ax[i,j].plot(true_parameters[0], true_parameters[1], color = "C2", marker = "o",  markersize = 5)    
        ax[i,j].plot(learned_parameters[0], learned_parameters[1], color = "r", marker = "x",  markersize = 5) 
        ax[i,j].set_xlim(xlim)
        ax[i,j].set_ylim(ylim)
        ax[i,j].set_title(f"n={npoints} noise={sigma}")

        x = mesh_1
        ax[i,j].plot(x, 9.81/2.5*x, "g--")

fig.suptitle(mode ,y=1.01)
plt.tight_layout()
        #ax[i,j].plot(x,2.5*9.81/2.5*x, "g--")

#plt.savefig(os.path.join(save_dir, "Ex2_posteriors_GT.png"), dpi=300,bbox_inches='tight' )

In [ ]:
mode = "PIGP"
run = 3
sigma_list = [0.0, 0.01, 0.1, 0.2]
npoints_list = [3, 5, 10, 25]
kernel_mode = "shared_kernel"
Al_mode = "withAl"
plt.rcParams.update({'font.size': 16})

fig, ax = plt.subplots(4,4, figsize = (12, 12))
N = 200
cmap = "jet"
for i in range(4):
    npoints = npoints_list[i]
    for j in range(4):
        sigma = sigma_list[j]
        data_ex2 = np.load(os.path.join(output_data_ex2, kernel_mode, Al_mode, f"Ex2_{mode}_n{npoints}_sigma{sigma:.2f}_{run}.npz"))
        learned_parameters = data_ex2["learned_parameters"]
        true_parameters = data_ex2["true_parameters"]
        laplace_covariance = data_ex2["laplace_covariance"]
        r1 = max(1, 1.1*abs(learned_parameters[0]-true_parameters[0]))
        r2 = max(1, 1.1*abs(learned_parameters[1]-true_parameters[1]))
        xlim = (learned_parameters[0]-r1, learned_parameters[0]+r1)
        ylim = (learned_parameters[1]-r2, learned_parameters[1]+r2)
        
        mesh_1 = torch.linspace(xlim[0], xlim[1], N)
        mesh_2 = torch.linspace(ylim[0], ylim[1], N)
        param1, param2 = torch.meshgrid(mesh_1, mesh_2, indexing="xy")
        
        mu = learned_parameters
        inv_cov = torch.inverse(torch.tensor(laplace_covariance[:2,:2]))
        param_vec = torch.stack([param1.flatten(), param2.flatten()], dim=-1)
        exponent_raw = torch.einsum('bi,ij,bj->b', param_vec - mu, inv_cov, param_vec - mu)
        prob_density = torch.exp(-0.5*(exponent_raw - exponent_raw.min()))
        prob_density /= prob_density.sum()*(mesh_1[1]-mesh_1[0])*(mesh_2[1]-mesh_2[0])
        prob_density = prob_density.reshape(N,N)
        
        surf = ax[i,j].pcolormesh(
                param1, param2, prob_density,
                cmap=cmap,
                shading="auto"
            )
        
        ax[i,j].plot(true_parameters[0], true_parameters[1], color = "C2", marker = "o",  markersize = 5)    
        ax[i,j].plot(learned_parameters[0], learned_parameters[1], color = "r", marker = "x",  markersize = 5) 
        ax[i,j].set_xlim(xlim)
        ax[i,j].set_ylim(ylim)
        ax[i,j].set_title(f"n={npoints} noise={sigma}")
        x = mesh_1
        ax[i,j].plot(x, 9.81/2.5*x, "g--")
        #ax[i,j].plot(x,2.5*9.81/2.5*x, "g--")
plt.tight_layout()

fig.suptitle(mode ,y=1.01)
plt.tight_layout()
        #ax[i,j].plot(x,2.5*9.81/2.5*x, "g--")

#plt.savefig(os.path.join(save_dir, "Ex2_posteriors_PIGP.png"), dpi=300,bbox_inches='tight' )

## Example where i think there's some higher resonance mode

In [ ]:
sigma = 0.001
run = 0

mode = "GT"
title = "GT"
num_tasks = 5
kernel_mode = "shared_kernel"
Al_mode = "withAl"
cmap = "jet"


for npoints in [5]:
    for sigma in [0.1]:
        fig = plt.figure(figsize=(18, 8), constrained_layout=True)
        if mode == "GT":
            task_labels = [r"$\theta_1$", r"$\theta_2$", r"$\theta_2$", r"$u$", r"$GT$"]
        else:
            task_labels = [r"$\theta_1$", r"$\theta_2$", r"$\theta_2$", r"$u$", r"$u$", r"$u$"]
        
        outer = gridspec.GridSpec(1, 2, width_ratios=[4,4], wspace=0.35)
        
        # ----------------- Load dataset -----------------
        data_ex2 = np.load(os.path.join(
                output_data_ex2, "shared_withAl",#kernel_mode, Al_mode,
                f"Ex2_{mode}_n{npoints}_sigma{sigma:.2f}_{run}.npz"), allow_pickle = True)
        
            
        for key in data_ex2:
            globals()[key] = data_ex2[key]
            #print(key)
        # ----------------- LEFT BLOCK: stacked time series -----------------
        inner = gridspec.GridSpecFromSubplotSpec(num_tasks, 1, subplot_spec=outer[0], hspace=0.05)
        for i in range(num_tasks):
            test_x_noi = test_x[:,0]
            test_index = test_x[:,-1]
            train_x_noi = train_x[:,0]
            train_index = train_x[:,-1]
            ax = fig.add_subplot(inner[i,0])
            ax.plot(test_x_noi[test_index == i], mean[test_index == i], 'b', linewidth=2)
            ax.fill_between(test_x_noi[test_index == i], lower[test_index == i], upper[test_index == i], alpha=0.3)
            ax.plot(test_x_noi[test_index == i], test_y[test_index == i], "r--")
            ax.plot(train_x_noi[train_index==i], train_y[train_index==i], 'k*')
            ax.set_ylabel(task_labels[i], fontsize=22)
        
            if i < num_tasks-1:
                ax.tick_params(labelbottom=False, bottom=False, labelsize=20)
            else:
                ax.set_xlabel(r"$t$", fontsize=22)
                ax.tick_params(labelsize=20)
            if i == 0:
                ax.set_title("Forward fit", fontsize=24)
        
        # ----------------- RIGHT BLOCK: parameter posterior -----------------
        ax_laplace = fig.add_subplot(outer[1])
        N = 400
        
        r1 = max(1, 1.1*abs(learned_parameters[0]-true_parameters[0]))
        r2 = max(1, 1.1*abs(learned_parameters[1]-true_parameters[1]))
        xlim = (learned_parameters[0]-r1, learned_parameters[0]+r1)
        ylim = (learned_parameters[1]-r2, learned_parameters[1]+r2)
        
        mesh_1 = torch.linspace(xlim[0], xlim[1], N)
        mesh_2 = torch.linspace(ylim[0], ylim[1], N)
        param1, param2 = torch.meshgrid(mesh_1, mesh_2, indexing="xy")
        
        mu = learned_parameters
        inv_cov = torch.inverse(torch.tensor(laplace_covariance[:2,:2]))
        param_vec = torch.stack([param1.flatten(), param2.flatten()], dim=-1)
        exponent_raw = torch.einsum('bi,ij,bj->b', param_vec - mu, inv_cov, param_vec - mu)
        prob_density = torch.exp(-0.5*(exponent_raw - exponent_raw.min()))
        prob_density /= prob_density.sum()*(mesh_1[1]-mesh_1[0])*(mesh_2[1]-mesh_2[0])
        prob_density = prob_density.reshape(N,N)
        
        surf = ax_laplace.pcolormesh(
                param1, param2, prob_density,
                cmap=cmap,
                shading="auto"
            )
        
        ax_laplace.plot(true_parameters[0], true_parameters[1], color = "C2", marker = "o",  markersize = 10)    
        ax_laplace.set_title("Parameter posterior", fontsize=24)
        x = mesh_1
        
        ax_laplace.plot(x, 9.81/2.5*x, "g--")
        ax_laplace.set_xlim(xlim)
        ax_laplace.set_ylim(ylim)
        fig.suptitle(f"{mode}, n = {npoints}, sigma = {sigma}", fontsize=26, y=1.02)
#plt.savefig(os.path.join(save_dir, "Ex2_example_mode.png"),dpi = 300, bbox_inches='tight' )